# AIvy Solver — Results Analysis

**What is AIvy?** A small experiment in using LLMs to synthesize the supporting invariants needed by the [Ivy](http://microsoft.github.io/ivy/) verifier. Every benchmark in `benchmarks/` ships two files: `ground_truth.ivy` (a known-good proof) and `stripped.ivy` (the same proof with some invariants deleted). The solver asks an LLM to fill them back in, runs `ivy_check` on the result, feeds any errors back, and retries until it either passes or hits `max_attempts`. Each end-to-end pass over the benchmark suite is one **run** and is dumped as a JSON file in `results/`.

This notebook aggregates every run in `results/`, joins each problem with its benchmark metadata, and renders five plots:

1. **Per-model summary** — overall success rate per model variant (`model × reasoning_effort`).
2. **Success rate vs. attempt budget** — how much of the success comes from the first try vs. the retry loop.
3. **Per-problem breakdown** — heatmap of every (model, problem) cell.
4. **Problem difficulty vs. success rate** — does benchmark size or # of missing invariants predict success?
5. **Failure-mode breakdown** — when an attempt fails, why does it fail?

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.max_rows = 100
pd.options.display.width = 140

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
BENCHMARKS_DIR = PROJECT_ROOT / "benchmarks"
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

assert RESULTS_DIR.exists(), f"results dir not found at {RESULTS_DIR}"
assert BENCHMARKS_DIR.exists(), f"benchmarks dir not found at {BENCHMARKS_DIR}"

## Loading helpers

Three flat dataframes drive every plot below:

- `problems_df` — one row per (run, problem), the unit for success-rate statistics.
- `attempts_df` — one row per individual LLM attempt, used for failure-mode breakdown.
- `metadata_df` — one row per benchmark, derived from `stripped.ivy` / `ground_truth.ivy`.

In [2]:
def load_runs(results_dir: Path) -> list[dict]:
    runs = []
    for path in sorted(results_dir.glob("*.json")):
        with open(path) as f:
            run = json.load(f)
        run["_file"] = path.name
        runs.append(run)
    return runs


def _short_model(model: str) -> str:
    return model.rsplit("/", 1)[-1]


def _variant_label(model: str, reasoning: str | None) -> str:
    return f"{_short_model(model)} [{reasoning or 'unset'}]"


def classify_attempt(attempt: dict) -> str:
    if attempt["passed"]:
        return "passed"
    out = (attempt.get("ivy_output") or "").lower()
    if out == "empty response":
        return "empty_response"
    if "modified" in out and "existing" in out:
        return "modified_existing_lines"
    if "timed out" in out or "timeout" in out:
        return "timeout"
    if "fail" in out or "error" in out:
        return "ivy_check_fail"
    return "other"


def build_problems_df(runs: list[dict]) -> pd.DataFrame:
    rows = []
    for run_idx, run in enumerate(runs):
        reasoning = run.get("reasoning_effort")
        variant = _variant_label(run["model"], reasoning)
        for p in run["problems"]:
            rows.append({
                "run_id": run_idx,
                "run_file": run["_file"],
                "run_timestamp": run["timestamp"],
                "model": run["model"],
                "reasoning_effort": reasoning,
                "model_variant": variant,
                "problem": p["problem_name"],
                "success": p["success"],
                "success_on_attempt": p["success_on_attempt"],
                "total_attempts": p["total_attempts"],
                "n_attempts_logged": len(p["attempts"]),
            })
    return pd.DataFrame(rows)


def build_attempts_df(runs: list[dict]) -> pd.DataFrame:
    rows = []
    for run_idx, run in enumerate(runs):
        reasoning_effort = run.get("reasoning_effort")
        variant = _variant_label(run["model"], reasoning_effort)
        for p in run["problems"]:
            for a in p["attempts"]:
                solution = a.get("llm_solution") or ""
                usage = a.get("usage") or {}
                cost = usage.get("cost")
                tokens = usage.get("total_tokens")
                ctd = usage.get("completion_tokens_details") or {}
                r_tokens = ctd.get("reasoning_tokens")
                rows.append({
                    "run_id": run_idx,
                    "model": run["model"],
                    "reasoning_effort": reasoning_effort,
                    "model_variant": variant,
                    "problem": p["problem_name"],
                    "attempt": a["attempt"],
                    "passed": a["passed"],
                    "outcome": classify_attempt(a),
                    "solution_chars": len(solution),
                    "cost_usd": float(cost) if isinstance(cost, (int, float)) else np.nan,
                    "total_tokens": int(tokens) if isinstance(tokens, (int, float)) else np.nan,
                    "reasoning_tokens": int(r_tokens) if isinstance(r_tokens, (int, float)) else np.nan,
                })
    return pd.DataFrame(rows)


def _count_invariants(text: str) -> int:
    return sum(1 for line in text.splitlines() if line.strip().startswith("invariant"))


def build_metadata_df(benchmarks_dir: Path) -> pd.DataFrame:
    rows = []
    for pdir in sorted(benchmarks_dir.iterdir()):
        stripped_path = pdir / "stripped.ivy"
        gt_path = pdir / "ground_truth.ivy"
        if not (pdir.is_dir() and stripped_path.exists() and gt_path.exists()):
            continue
        stripped = stripped_path.read_text()
        gt = gt_path.read_text()
        rows.append({
            "problem": pdir.name,
            "stripped_chars": len(stripped),
            "stripped_lines": stripped.count("\n") + 1,
            "ground_truth_chars": len(gt),
            "ground_truth_lines": gt.count("\n") + 1,
            "invariants_stripped": _count_invariants(stripped),
            "invariants_ground_truth": _count_invariants(gt),
            "invariants_to_find": _count_invariants(gt) - _count_invariants(stripped),
        })
    return pd.DataFrame(rows)

In [3]:
runs = load_runs(RESULTS_DIR)
problems_df = build_problems_df(runs)
attempts_df = build_attempts_df(runs)
metadata_df = build_metadata_df(BENCHMARKS_DIR)

print(
    f"{len(runs)} run(s) | "
    f"{problems_df['model_variant'].nunique()} model variant(s) | "
    f"{problems_df['problem'].nunique()} problem(s) | "
    f"{len(attempts_df):,} individual LLM attempts"
)

21 run(s) | 21 model variant(s) | 58 problem(s) | 4,835 individual LLM attempts


## 1. Per-model summary

One bar per model variant (`model × reasoning_effort`). Bar height is the success rate — fraction of `(run, problem)` pairs the variant solved. Whiskers are Wilson 95% confidence intervals, which tighten with more runs. Use this section to compare overall capability and to read off the marginal effect of bumping `reasoning_effort` while holding the underlying model fixed.

In [4]:
def wilson_ci(p: float, n: int, z: float = 1.96) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion. Returns (lower, upper).

    Better-behaved than the normal-approximation SE when p is near 0 or 1, or n is small:
    bounds stay inside [0, 1] and the interval is asymmetric near the boundaries.
    """
    if n == 0:
        return 0.0, 0.0
    denom = 1.0 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z / denom) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return float(max(0.0, center - half)), float(min(1.0, center + half))


def add_ci_bounds(df: pd.DataFrame, p_col: str, n_col: str) -> pd.DataFrame:
    """Attach Wilson 95% CI bounds and plotly-friendly asymmetric error deltas."""
    bounds = [wilson_ci(p, n) for p, n in zip(df[p_col], df[n_col])]
    df["ci_low"] = [b[0] for b in bounds]
    df["ci_high"] = [b[1] for b in bounds]
    df["err_minus"] = df[p_col] - df["ci_low"]
    df["err_plus"] = df["ci_high"] - df[p_col]
    return df


summary = (
    problems_df.groupby(["model_variant", "model", "reasoning_effort"], dropna=False)
    .agg(
        n_rows=("success", "size"),
        n_passed=("success", "sum"),
        success_rate=("success", "mean"),
        mean_attempts_on_success=(
            "success_on_attempt",
            lambda s: float(s.dropna().mean()) if s.dropna().size else np.nan,
        ),
        mean_attempts_used=("total_attempts", "mean"),
    )
    .reset_index()
)
summary = add_ci_bounds(summary, "success_rate", "n_rows")

In [5]:
fig = px.bar(
    summary.sort_values(["model", "reasoning_effort"]),
    x="model_variant",
    y="success_rate",
    color="model",
    error_y="err_plus",
    error_y_minus="err_minus",
    text=summary["success_rate"].map(lambda v: f"{v:.0%}"),
    title="Overall success rate per model variant",
    hover_data=["model", "reasoning_effort", "n_rows", "n_passed", "ci_low", "ci_high"],
)
fig.update_layout(
    yaxis_tickformat=",.0%",
    yaxis_range=[0, 1],
    width=900,
    height=480,
    xaxis_title="model_variant",
)
fig.write_image(PLOTS_DIR / "01_overall_success_rate.png", scale=2)
fig.show()

/tmp/ipykernel_1199/1740383020.py:19: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "01_overall_success_rate.png", scale=2)


### 1b. Reasoning-effort comparison (within model)

Bars are grouped per model so you can read off the marginal effect of changing `reasoning_effort` while holding the underlying model fixed.

In [6]:
reasoning_compare = summary.copy()
reasoning_compare["reasoning_effort"] = reasoning_compare["reasoning_effort"].fillna("unset")

fig = px.bar(
    reasoning_compare.sort_values(["model", "reasoning_effort"]),
    x="model",
    y="success_rate",
    color="reasoning_effort",
    barmode="group",
    error_y="err_plus",
    error_y_minus="err_minus",
    text=reasoning_compare["success_rate"].map(lambda v: f"{v:.0%}"),
    title="Reasoning effort comparison within each model",
    hover_data=["model_variant", "n_rows", "n_passed", "ci_low", "ci_high"],
)
fig.update_layout(yaxis_tickformat=",.0%", yaxis_range=[0, 1], width=900, height=450)
fig.write_image(PLOTS_DIR / "02_reasoning_effort_compare.png", scale=2)
fig.show()

/tmp/ipykernel_1199/4185117019.py:17: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "02_reasoning_effort_compare.png", scale=2)


## 2. Success rate vs. attempt budget

Each problem gets up to `max_attempts` tries before the solver gives up. This plot asks: _how does success rate trade off against the attempt budget?_ The point at $k=1$ is the one-shot success rate; the curve plateaus at $k = $ `max_attempts`. The slope between the two tells you how much the retry-with-error-feedback loop is actually buying you over single-shot prompting.

In [7]:
def success_rate_by_budget(df: pd.DataFrame) -> pd.DataFrame:
    max_budget = int(df["total_attempts"].max()) if len(df) else 0
    rows = []
    for variant, g in df.groupby("model_variant"):
        n = len(g)
        succ_at = g.loc[g["success"], "success_on_attempt"].astype(int)
        for k in range(0, max_budget + 1):
            p = float((succ_at <= k).sum()) / n if n else 0.0
            rows.append({"model_variant": variant, "budget": k, "success_rate": p, "n": n})
    return add_ci_bounds(pd.DataFrame(rows), "success_rate", "n")


budget_df = success_rate_by_budget(problems_df)

In [8]:
def line_with_error_band(df: pd.DataFrame, x: str, y: str, lower: str, upper: str, color: str, title: str) -> go.Figure:
    fig = px.line(df, x=x, y=y, color=color, markers=True, title=title)
    for trace in list(fig.data):
        sub = df[df[color] == trace.name].sort_values(x)
        xs = sub[x].to_list()
        ys_upper = sub[upper].clip(upper=1.0).to_list()
        ys_lower = sub[lower].clip(lower=0.0).to_list()
        rgb = trace.line.color.lstrip("#")
        r, g, b = (int(rgb[i : i + 2], 16) for i in (0, 2, 4))
        fig.add_trace(
            go.Scatter(
                x=xs + xs[::-1],
                y=ys_upper + ys_lower[::-1],
                fill="toself",
                fillcolor=f"rgba({r},{g},{b},0.2)",
                line=dict(color="rgba(255,255,255,0)"),
                hoverinfo="skip",
                showlegend=False,
                legendgroup=trace.name,
            )
        )
    fig.update_layout(yaxis_tickformat=",.0%", yaxis_range=[0, 1.02], width=900, height=520)
    return fig


fig = line_with_error_band(
    budget_df,
    x="budget",
    y="success_rate",
    lower="ci_low",
    upper="ci_high",
    color="model_variant",
    title="Success rate vs. number of attempts",
)
fig.update_xaxes(dtick=1, title="attempts (k)")
fig.write_image(PLOTS_DIR / "03_success_rate_vs_attempts.png", scale=2)
fig.show()

/tmp/ipykernel_1199/3062638704.py:36: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "03_success_rate_vs_attempts.png", scale=2)


## 3. Per-problem breakdown

Heatmap of `success_rate` with rows = model variants and columns = problems. Rows are sorted weakest → strongest model bottom-to-top; columns are sorted hardest → easiest left-to-right. Vertical red/green bands flag universally-hard / universally-easy problems; horizontal bands flag uniformly weak / strong models; off-diagonal hot- or cold-spots are model-specific blind spots worth investigating individually.

In [9]:
per_problem = (
    problems_df.groupby(["model_variant", "model", "reasoning_effort", "problem"], dropna=False)
    .agg(
        n_runs=("success", "size"),
        success_rate=("success", "mean"),
        mean_attempts_on_success=(
            "success_on_attempt",
            lambda s: float(s.dropna().mean()) if s.dropna().size else np.nan,
        ),
    )
    .reset_index()
)
per_problem = add_ci_bounds(per_problem, "success_rate", "n_runs")

In [10]:
heatmap_df = per_problem.pivot_table(
    index="model_variant", columns="problem", values="success_rate", aggfunc="mean"
)
problem_order = heatmap_df.mean(axis=0).sort_values().index
variant_order = heatmap_df.mean(axis=1).sort_values().index
heatmap_df = heatmap_df.loc[variant_order, problem_order]

fig = px.imshow(
    heatmap_df,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=1,
    aspect="auto",
    labels=dict(color="success rate", x="problem (sorted hardest → easiest)", y="model variant"),
    title="Per-problem success rate (rows: model variants, columns: problems)",
)
fig.update_layout(
    width=1100,
    height=max(360, 36 * len(heatmap_df.index)),
    xaxis_tickangle=-45,
    coloraxis_colorbar=dict(tickformat=",.0%"),
)
fig.update_xaxes(tickfont=dict(size=9))
fig.write_image(PLOTS_DIR / "04_per_problem_heatmap.png", scale=2)
fig.show()

/tmp/ipykernel_1199/1047298546.py:24: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "04_per_problem_heatmap.png", scale=2)


## 4. Problem difficulty vs. success rate

Joins per-problem success with benchmark metadata. We try two difficulty proxies:

- `invariants_to_find` — number of invariants deleted from `ground_truth.ivy` to produce `stripped.ivy`. The most direct measure of how much the LLM has to do.
- `stripped_chars` — raw size of the input the LLM sees. A weaker proxy, but easy to compute.

Each marker is one problem, with success rate averaged across all model variants (per-model variation is already in section 3). The dashed line is an OLS fit on the per-problem aggregate. The title reports Spearman ρ — a rank correlation that captures any monotonic relationship and is robust to outliers.

In [11]:
difficulty_df = per_problem.merge(metadata_df, on="problem", how="left")

In [12]:
def difficulty_scatter(df: pd.DataFrame, x: str, title: str) -> go.Figure:
    agg = (
        df.dropna(subset=[x])
        .groupby("problem", as_index=False)
        .agg(success_rate=("success_rate", "mean"), n_runs=("n_runs", "sum"), **{x: (x, "first")})
    )
    if len(agg) >= 2:
        rho = float(agg[x].rank().corr(agg["success_rate"].rank()))
        title = f"{title}  (Spearman ρ = {rho:+.2f}, n = {len(agg)} problems)"
    fig = px.scatter(
        agg,
        x=x,
        y="success_rate",
        size="n_runs",
        hover_data=["problem", "n_runs"],
        title=title,
    )
    if len(agg) >= 2 and agg[x].nunique() >= 2:
        coef = np.polyfit(agg[x], agg["success_rate"], 1)
        xs = np.linspace(agg[x].min(), agg[x].max(), 50)
        ys = np.clip(np.polyval(coef, xs), 0.0, 1.0)
        fig.add_trace(
            go.Scatter(x=xs, y=ys, mode="lines", name="OLS fit", line=dict(dash="dash", color="black"))
        )
    fig.update_layout(yaxis_tickformat=",.0%", yaxis_range=[0, 1.05], width=850, height=480)
    return fig


fig = difficulty_scatter(difficulty_df, "invariants_to_find", "Success rate vs. # invariants to synthesize")
fig.write_image(PLOTS_DIR / "05_difficulty_vs_invariants.png", scale=2)
fig.show()

fig = difficulty_scatter(difficulty_df, "stripped_chars", "Success rate vs. stripped program size (chars)")
fig.write_image(PLOTS_DIR / "06_difficulty_vs_program_size.png", scale=2)
fig.show()

/tmp/ipykernel_1199/1932786106.py:30: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "05_difficulty_vs_invariants.png", scale=2)


/tmp/ipykernel_1199/1932786106.py:34: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "06_difficulty_vs_program_size.png", scale=2)


## 5. Failure-mode breakdown

Stacked bar of every individual attempt's outcome, per model variant. Useful for spotting systematic issues. Categories come from `classify_attempt` near the top of the notebook:

- `passed` — accepted by `ivy_check`.
- `ivy_check_fail` — Ivy ran but rejected the candidate invariants.
- `timeout` — `ivy_check` exceeded `ivy_check_timeout`.
- `modified_existing_lines` — the LLM rewrote pre-existing lines instead of only adding invariants (prompt-compliance failure).
- `empty_response` — the LLM returned nothing extractable (prompting / parsing failure).
- `other` — unmatched by the rules above; usually worth eyeballing manually.

In [13]:
outcome_order = ["passed", "ivy_check_fail", "timeout", "modified_existing_lines", "empty_response", "other"]

outcome_counts = (
    attempts_df.groupby(["model_variant", "outcome"])
    .size()
    .reset_index(name="count")
)
outcome_counts["outcome"] = pd.Categorical(outcome_counts["outcome"], categories=outcome_order, ordered=True)
outcome_counts = outcome_counts.sort_values(["model_variant", "outcome"])

In [14]:
fig = px.bar(
    outcome_counts,
    x="model_variant",
    y="count",
    color="outcome",
    category_orders={"outcome": outcome_order},
    title="Attempt outcomes per model variant",
)
fig.update_layout(width=950, height=480, barmode="stack", xaxis_title="model_variant")
fig.write_image(PLOTS_DIR / "07_attempt_outcomes.png", scale=2)
fig.show()

/tmp/ipykernel_1199/4129539194.py:10: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "07_attempt_outcomes.png", scale=2)


## 6. Cost & efficiency

Every solve attempt carries `usage.cost` (USD billed by OpenRouter) and `usage.total_tokens`. That lets us ask the obvious money-conscious question: _which model variant solves the most problems per dollar (or per million tokens) spent?_

Two complementary views below:

1. **Efficiency ranking** — solves per $1 and solves per 1M tokens, both sorted high → low. Same data, two y-axes; ranks can disagree because models charge different rates per token.
2. **Cost vs. quality Pareto** — total spend (log scale) vs. overall success rate, with the Pareto frontier drawn through the dominating variants. Points _below_ the frontier are strictly dominated: there is some other variant that is both cheaper and more accurate.

The unit of "solve" here is one `(run, problem)` pair the variant got past `ivy_check` — i.e. the same numerator we used for success rate in section 1. The denominator is total spend across **all** attempts (passing _and_ failing), since that's what you actually paid.

In [15]:
spend = (
    attempts_df.groupby(["model_variant", "model", "reasoning_effort"], dropna=False)
    .agg(
        n_attempts=("attempt", "size"),
        total_cost_usd=("cost_usd", "sum"),
        total_tokens=("total_tokens", "sum"),
    )
    .reset_index()
)
solved_by_variant = (
    problems_df.groupby("model_variant")["success"].sum().reset_index(name="n_solved")
)
spend = spend.merge(solved_by_variant, on="model_variant", how="left")
spend = spend.merge(summary[["model_variant", "success_rate"]], on="model_variant", how="left")
spend["solves_per_dollar"] = spend["n_solved"] / spend["total_cost_usd"].replace(0, np.nan)
spend["solves_per_million_tokens"] = spend["n_solved"] / (spend["total_tokens"] / 1e6).replace(0, np.nan)

by_dollar = spend.sort_values("solves_per_dollar", ascending=True)
by_token = spend.sort_values("solves_per_million_tokens", ascending=True)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Problems solved per $1", "Problems solved per 1M tokens"),
    horizontal_spacing=0.22,
)
fig.add_trace(
    go.Bar(
        x=by_dollar["solves_per_dollar"],
        y=by_dollar["model_variant"],
        orientation="h",
        marker_color="#2E86AB",
        text=[f"{v:.1f}" for v in by_dollar["solves_per_dollar"]],
        textposition="outside",
        customdata=by_dollar[["n_solved", "total_cost_usd"]].values,
        hovertemplate="<b>%{y}</b><br>%{customdata[0]:.0f} solves / $%{customdata[1]:.2f}<br>= %{x:.2f} solves / $1<extra></extra>",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(
        x=by_token["solves_per_million_tokens"],
        y=by_token["model_variant"],
        orientation="h",
        marker_color="#A23B72",
        text=[f"{v:.1f}" for v in by_token["solves_per_million_tokens"]],
        textposition="outside",
        customdata=by_token[["n_solved", "total_tokens"]].values,
        hovertemplate="<b>%{y}</b><br>%{customdata[0]:.0f} solves / %{customdata[1]:,.0f} tokens<br>= %{x:.2f} solves / 1M tokens<extra></extra>",
    ),
    row=1, col=2,
)
fig.update_layout(
    width=1100,
    height=520,
    showlegend=False,
    title="Bang for buck: problems solved per unit of spend",
    margin=dict(l=10, r=10, t=80, b=40),
)
fig.update_xaxes(title_text="solves / $1", row=1, col=1)
fig.update_xaxes(title_text="solves / 1M tokens", row=1, col=2)
fig.write_image(PLOTS_DIR / "08_solves_per_spend.png", scale=2)
fig.show()

/tmp/ipykernel_1199/1657739089.py:61: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "08_solves_per_spend.png", scale=2)


In [16]:
pareto = spend.sort_values("total_cost_usd").copy()
best_so_far = -np.inf
on_pareto = []
for sr in pareto["success_rate"]:
    if sr > best_so_far:
        best_so_far = sr
        on_pareto.append(True)
    else:
        on_pareto.append(False)
pareto["on_pareto"] = on_pareto

fig = px.scatter(
    spend,
    x="total_cost_usd",
    y="success_rate",
    color="model",
    size="n_attempts",
    text="model_variant",
    log_x=True,
    hover_data={
        "model_variant": False,
        "n_solved": True,
        "n_attempts": True,
        "total_cost_usd": ":.2f",
        "total_tokens": ":,",
        "success_rate": ":.1%",
    },
    title="Cost vs. quality Pareto: total spend vs. overall success rate",
)
fig.update_traces(textposition="top center", textfont_size=10)
frontier = pareto[pareto["on_pareto"]].sort_values("total_cost_usd")
fig.add_trace(
    go.Scatter(
        x=frontier["total_cost_usd"],
        y=frontier["success_rate"],
        mode="lines",
        name="Pareto frontier",
        line=dict(color="black", dash="dash", width=2),
        hoverinfo="skip",
    )
)
fig.update_layout(
    width=950,
    height=560,
    yaxis_tickformat=",.0%",
    yaxis_range=[0, 1],
    xaxis_title="total cost (USD, log scale)",
    yaxis_title="overall success rate",
)
fig.write_image(PLOTS_DIR / "09_cost_quality_pareto.png", scale=2)
fig.show()

/tmp/ipykernel_1199/293893945.py:50: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(PLOTS_DIR / "09_cost_quality_pareto.png", scale=2)
